# 04 - VAE-LSTM + k-means (calendar weeks, no overlap)

Last test of the thesis: the same VAE-LSTM, trained on one scalogram per **calendar week** instead of one per day. It trains the model and saves its regimes; it does not analyze them. Every metric is computed afterwards, by week, by `scripts/07_calweek_metrics.py`.

How this differs from notebook 03:
- **Data.** Notebook 03 has one weekly scalogram per day (day t and the 6 days before it), so two consecutive images share 6 of their 7 days. Here each image is one calendar week, Monday 00:00 to Sunday 23:00 (168 hours), and consecutive images do not overlap: 1386 weeks from 2000-01-03 to 2026-07-26, in `data/processed/scalograms/real_price_log_calweek/` (`scripts/04_generate_scalograms.py --price log --window calweek`).
- **The model sees only the images.** The input of the VAE-LSTM is the 128 x 168 scalogram of each week, with the mean of the week removed, so it shows the shape of the price over the 168 hours (including the changes from one day to the next) but not its level.
- **Monthly reading.** The LSTM reads 4 consecutive weeks (`seq_len = 4`, about one month), and the vector of week t comes from that week and the 3 before it. Training uses every run of 4 weeks inside a period (`stride = 1`), because there are only 770 training weeks.
- **Two clustering inputs per model.** Each trained model is clustered twice, for k = 3, 4 and 5:
  - `latents + level` (study `calweek`): the vector plus the mean and standard deviation of the log price of the week, z-scored on training weeks, as in every earlier notebook. It shows whether the image adds anything to the price level.
  - `latents` (study `calweek_latents_only`): the vector alone, so the regimes come only from the images.
- **beta and lambda.** The image has 7 times more pixels than a daily one, so the KL and smoothness terms are scaled: lambda = 700 and beta = 1 and 7, 3 seeds.

Loss = reconstruction + beta * KL + lambda * smoothness, where smoothness is the squared change of the latent mean between consecutive weeks relative to its spread (Nie et al., 2026).

Pipeline: 4 weekly scalograms -> CNN per week -> causal LSTM -> latent mean per week (+ mean and std of the log price of the week, for the first input) -> k-means -> regime

Methodology notes:
1. Weeks are split by the date of their Monday (`val_start` and `test_start` in `src/data.py`): 770 train weeks (before 2014-10-01), 326 validation weeks (2014-10-01 to 2020-12-31) and 290 test weeks (from 2021-01-01).
2. K-means is fitted on training weeks only and then assigns every week. Regimes are numbered by ascending mean real price of their weeks, so the highest number is the most expensive regime. The price of a week is the mean of its 7 daily real prices; it numbers the regimes and is used to measure them, never as an input of the VAE-LSTM.
3. `regimes.csv` has one row per week, dated by its Monday.
4. A week is El Nino when the ONI of the month of its Thursday is >= 0.5 (the month that holds at least 4 of its 7 days).

<d.benavidess@uniandes.edu.co>


In [ ]:
import sys
sys.path.append("..")

import itertools
import json
from pathlib import Path

import torch

import numpy as np
import altair as alt
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, date_split, val_start, test_start, ScalogramSequenceDataset
from src.models.vae_lstm import VAELSTM
from src.training import set_seed, train, active_dims
from src.clustering import with_level, level_columns
from src.plots import loss_curves, plot_reconstructions, regime_timeline
from src.runs import prepare_runs, save_all, cluster_runs, tsne_grid, regime_strips
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment configuration

Change only this cell between runs.
- `grid` holds the values of `lam`, `beta` and `seed`. The model is trained once for every combination and clustered for every k in `k_values`, with the two inputs. The run folder of each combination ends with its values, e.g. `lam700_beta1_seed42`. The other hyperparameters are in `config` and are the same for every combination.
- `k_values` is always 3, 4 and 5.
- `study` names the folder `experiments/vae_lstm/<study>/` of the `latents + level` runs, which hold the trained model (`artifacts/`); `latents_only_study` names the folder of the `latents` runs, whose `trained_in` points to the `study` run with k = 3.
- `tag` is an optional short label added to the run folder name before the grid values. Leave it empty to omit it.
- `load_from` is the `run_id` of an earlier run of `study`. When set, training is skipped, the grid is ignored and that run's model, history and latents are clustered again with both inputs; its hyperparameters are used, so only `k_values`, `study` and `tag` come from this cell. An empty `tag` takes the tag of the loaded run.
- `active_threshold`: a latent dim is active when its KL on the training weeks at the last epoch is at least this value (nats).


In [ ]:
scalogram_set = "real_price_log_calweek"
k_values = [3, 4, 5]
study = "calweek"
latents_only_study = "calweek_latents_only"
tag = ""
load_from = None
active_threshold = 0.75

grid = {
    "lam": [100.0, 700.0],
    "beta": [1.0, 7.0],
    "seed": [42, 43, 44],
}

config = {
    "model": "vae_lstm",
    "study": study,
    "scalogram_set": scalogram_set,
    "k_values": k_values,
    "clustering_input": "latents + level",
    "z_dim": 16,
    "filters": [128, 64, 32],
    "proj_dim": 128,
    "rnn_type": "lstm",
    "rnn_hidden": 64,
    "seq_len": 4,
    "stride": 1,
    "kl_warmup_epochs": 5,
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 16,
    "epochs": 30,
    "val_start": val_start,
    "test_start": test_start,
}

## 1. Combinations

Every combination of the values in `grid`, clustered with both inputs. Its run folders are created when its training starts. Section 4 shows the loss curves of every model and the reconstructions of the models with seed 42, and section 5 their regimes (or those of the loaded model).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if load_from and not tag:
    tag = json.loads((exp.EXPERIMENTS_DIR / load_from / "config.json").read_text()).get("tag", "")
combos = [{}] if load_from else [dict(zip(grid, values)) for values in itertools.product(*grid.values())]
print(f"{len(combos)} trainings x {len(k_values)} k = {len(combos) * len(k_values)} runs")
print(f"Device: {device}")

## 2. Data

In [ ]:
scalogram_dir = Path("../data/processed/scalograms") / config["scalogram_set"]
data_dir = Path("../data/processed")

images, metadata = load_scalograms(scalogram_dir)
context = load_context(data_dir, metadata)
train_idx, val_idx, test_idx = date_split(context["date"], config["val_start"], config["test_start"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

img_c, img_h, img_w = images.shape[1:]
print(f"Scalograms ({config['scalogram_set']}): {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")
print(f"Level columns (mean and std of the log price of the week): {level_columns(metadata)}")
print(f"Weeks: {len(context)}, from {context['date'].iloc[0]:%Y-%m-%d} (Monday) to {context['date'].iloc[-1]:%Y-%m-%d} (Monday)")
splits = exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx})
print(splits)

In [ ]:
train_set = ScalogramSequenceDataset(images, train_idx, config["seq_len"], config["stride"])
val_set = ScalogramSequenceDataset(images, val_idx, config["seq_len"], config["stride"])
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)
print(f"Sequences of {config['seq_len']} weeks: train {len(train_set)}, val {len(val_set)}")

In [ ]:
def bce_floor(x, chunk=500):
    """Computed in blocks of `chunk` weeks."""
    floors = []
    for start in range(0, len(x), chunk):
        block = np.clip(x[start:start + chunk].reshape(-1, x[0].size).astype(np.float64), 1e-7, 1 - 1e-7)
        floors.append(-(block * np.log(block) + (1 - block) * np.log(1 - block)).sum(axis=1))
    return np.concatenate(floors)

for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    f = bce_floor(images[idx])
    print(f"{name}: floor {f.mean():.0f}, mean pixel {images[idx].mean():.3f}")

## 3. Training and clustering

For every combination: build the model, train it (skipped with `load_from`), extract the latent vectors and cluster once for each k and each input. Each run folder of `study` (`latents + level`) gets:
- `config.json`, `history.json`, `regimes.csv` (one row per week, see `runs.regime_table`), `metrics.json` (silhouette, Davies-Bouldin and Calinski-Harabasz on the clustering features) and its row in `registry.csv`
- `tables/`: hyperparameters, regime profile (weeks, share, real price, reservoir, ONI and El Nino share of each regime) and clustering metrics
- `figures/`: loss curves, reconstructions of four test weeks, t-SNE and regime timeline
- `artifacts/` (not in git): model weights and latent vectors

Each run folder of `latents_only_study` (`latents`) gets the same files except `history.json`, the loss curves, the reconstructions and `artifacts/`: its model is the one in the folder named by `trained_in`.

The printed line shows the KL of each latent dim at the last epoch (`kl_dim0` is the first dim of the vector, and so on) and how many dims are active (KL >= `active_threshold`).


In [ ]:
shown, trained_runs = {}, {}
for i, combo in enumerate(combos, 1):
    run_tag = "_".join(filter(None, [tag] + [f"{key}{value:g}" for key, value in combo.items()]))
    run_config, run_dirs, run_configs, trained = prepare_runs({**config, **combo, "tag": run_tag}, load_from, device)
    set_seed(run_config["seed"])
    trained_in = run_configs[k_values[0]]["trained_in"]
    show_model = bool(load_from) or run_config["seed"] == 42
    print(f"[{i}/{len(combos)}] {run_tag or load_from}: trained_in {trained_in}")

    model = VAELSTM(run_config["z_dim"], img_c, img_h, img_w, run_config["filters"],
                        run_config["proj_dim"], run_config["rnn_hidden"], run_config["rnn_type"]).to(device)
    if trained:
        model.load_state_dict(trained["state_dict"])
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    for k, run_dir in run_dirs.items():
        run_configs[k]["n_params"] = n_params
        exp.save_config(run_dir, run_configs[k])

    if trained:
        history = trained["history"]
    else:
        history = train(model, train_loader, val_loader, device, epochs=run_config["epochs"], lr=run_config["lr"],
                        beta=run_config["beta"], kl_warmup_epochs=run_config["kl_warmup_epochs"],
                        extra_loss_kwargs={"lam": run_config["lam"]})
    save_all(run_dirs, exp.save_history, history)
    save_all(run_dirs, exp.save_model, model)
    loss_plot = loss_curves(history, components=("total", "recon", "kl", "smooth"))
    save_all(run_dirs, lambda d: loss_plot.save(exp.figure_path(d, "loss_curves"), scale_factor=2.0))
    trained_runs[trained_in] = {"loss_curves": loss_plot}

    model.eval()
    start = test_idx[len(test_idx) // 2]
    seq = torch.from_numpy(images[start:start + run_config["seq_len"]]).unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model(seq)[0][0].cpu().numpy()
    shown_weeks = np.arange(run_config["seq_len"])  # the 4 weeks of one sequence
    fig = plot_reconstructions(images[start + shown_weeks], recon[shown_weeks],
                               [f"week {d.date()}" for d in context["date"].iloc[start + shown_weeks]],
                               n_show=len(shown_weeks))
    save_all(run_dirs, lambda d: fig.savefig(exp.figure_path(d, "reconstructions"), dpi=150, bbox_inches="tight"))
    if show_model:
        trained_runs[trained_in]["reconstructions"] = fig
    plt.close(fig)

    latents = trained["latents"] if trained else model.extract_latents(images, device, seq_len=run_config["seq_len"])
    save_all(run_dirs, exp.save_latents, latents)
    n_active, kl_dims = active_dims(history, active_threshold)
    print(f"    KL per dim {kl_dims}, active dims {n_active} of {run_config['z_dim']}")

    features = with_level(latents, metadata, train_idx)
    rng = np.random.default_rng(run_config["seed"])
    plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
    coords = TSNE(n_components=2, perplexity=30, random_state=run_config["seed"]).fit_transform(features[plot_idx])
    results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, coords, plot_idx)

    # the same latents clustered alone: the regimes come only from the images
    latents_config = {**run_config, "study": latents_only_study, "clustering_input": "latents"}
    _, lo_dirs, lo_configs, _ = prepare_runs(latents_config, load_from, device, trained_in=trained_in)
    for k, run_dir in lo_dirs.items():
        lo_configs[k]["n_params"] = n_params
        exp.save_config(run_dir, lo_configs[k])
    lo_features = latents.astype(np.float64)
    lo_coords = TSNE(n_components=2, perplexity=30, random_state=run_config["seed"]).fit_transform(lo_features[plot_idx])
    lo_results = cluster_runs(lo_dirs, lo_configs, lo_features, context, train_idx, split_dates, lo_coords, plot_idx)
    print(f"    latents + level -> {exp.run_id(run_dirs[k_values[0]]).rsplit('/', 1)[0]}, "
          f"latents -> {exp.run_id(lo_dirs[k_values[0]]).rsplit('/', 1)[0]}")
    if show_model:
        shown[f"{run_tag or load_from} (latents + level)"] = {"results": results, "coords": coords, "plot_idx": plot_idx}
        shown[f"{run_tag or load_from} (latents)"] = {"results": lo_results, "coords": lo_coords, "plot_idx": plot_idx}

## 4. Loss curves and reconstructions

First the loss curves of every trained model (total, reconstruction, KL and smoothness on training and validation weeks), each under the `run_id` of the run where it was trained. Then the reconstructions of four consecutive test weeks of the models with seed 42 (or of the loaded model): original scalograms on top, reconstructed below. Both figures are also saved in `figures/` of every `study` run of the model.


In [ ]:
for run_id, figures in trained_runs.items():
    print(run_id)
    display(figures["loss_curves"])

for run_id, figures in trained_runs.items():
    if "reconstructions" in figures:
        print(run_id)
        display(figures["reconstructions"])

## 5. Regimes of the models with seed 42

For each shown model and each input (`latents + level` and `latents`): the t-SNE of the clustering features colored by the regimes of k = 3, 4 and 5, the regime timeline of each k (real price of each week colored by regime, regime strip and reservoir level, with El Nino periods shaded and the split boundaries dashed), and the regime strips of the three k together.


In [ ]:
for name, s in shown.items():
    print(name)
    display(tsne_grid(s["coords"], s["plot_idx"], s["results"]))
    for k, ctx in s["results"].items():
        display(regime_timeline(ctx, split_dates, title=f"{name}: detected regimes over time (k={k})"))
    display(regime_strips(s["results"]))